In [1]:
print("hello")

hello


In [14]:
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="../.env")
DATABASE_URL = os.getenv("DATABASE_URL")

conn = psycopg2.connect(DATABASE_URL)
df = pd.read_sql("SELECT * FROM aqi_processed", conn)
conn.close()

C:\Users\DEEPAK\AppData\Local\Temp\ipykernel_19072\2148131568.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM aqi_processed", conn)


In [15]:
df.head()

,id,station,last_update,CO,NH3,NO2,OZONE,PM10,PM2.5,SO2,processed_at
0,1,"Alipur, Delhi - DPCC",02-04-2026 16:00:00,28.0,14.0,27.0,47.0,135.0,131.0,11.0,2026-04-02 23:10:46.525035
1,2,"Anand Vihar, Delhi - DPCC",02-04-2026 16:00:00,68.0,6.0,31.0,51.0,248.0,220.0,28.0,2026-04-02 23:10:46.525035
2,3,"Ashok Vihar, Delhi - DPCC",02-04-2026 16:00:00,38.0,7.0,51.0,151.0,167.0,173.0,22.0,2026-04-02 23:10:46.525035
3,4,"Bawana, Delhi - DPCC",02-04-2026 16:00:00,65.0,7.0,27.0,202.0,171.0,173.0,19.0,2026-04-02 23:10:46.525035
4,5,"Burari Crossing, Delhi - IMD",02-04-2026 16:00:00,29.0,NaN,36.0,177.0,150.0,185.0,NaN,2026-04-02 23:10:46.525035


In [16]:
# clean station name - extract only area name
df["station_clean"] = df["station"].str.split(",").str[0].str.strip()

print(df[["station", "station_clean"]].head(10))

                                        station  \
0                          Alipur, Delhi - DPCC   
1                     Anand Vihar, Delhi - DPCC   
2                     Ashok Vihar, Delhi - DPCC   
3                          Bawana, Delhi - DPCC   
4                  Burari Crossing, Delhi - IMD   
5                CRRI Mathura Road, Delhi - IMD   
6                Cantonment Area, Delhi - DPCC    
7                   Chandni Chowk, Delhi - IITM   
8     Commonwealth Sports Complex, Delhi - DPCC   
9  Dr. Karni Singh Shooting Range, Delhi - DPCC   

                    station_clean  
0                          Alipur  
1                     Anand Vihar  
2                     Ashok Vihar  
3                          Bawana  
4                 Burari Crossing  
5               CRRI Mathura Road  
6                 Cantonment Area  
7                   Chandni Chowk  
8     Commonwealth Sports Complex  
9  Dr. Karni Singh Shooting Range  


In [17]:
df.head()

,id,station,last_update,CO,NH3,NO2,OZONE,PM10,PM2.5,SO2,processed_at,station_clean
0,1,"Alipur, Delhi - DPCC",02-04-2026 16:00:00,28.0,14.0,27.0,47.0,135.0,131.0,11.0,2026-04-02 23:10:46.525035,Alipur
1,2,"Anand Vihar, Delhi - DPCC",02-04-2026 16:00:00,68.0,6.0,31.0,51.0,248.0,220.0,28.0,2026-04-02 23:10:46.525035,Anand Vihar
2,3,"Ashok Vihar, Delhi - DPCC",02-04-2026 16:00:00,38.0,7.0,51.0,151.0,167.0,173.0,22.0,2026-04-02 23:10:46.525035,Ashok Vihar
3,4,"Bawana, Delhi - DPCC",02-04-2026 16:00:00,65.0,7.0,27.0,202.0,171.0,173.0,19.0,2026-04-02 23:10:46.525035,Bawana
4,5,"Burari Crossing, Delhi - IMD",02-04-2026 16:00:00,29.0,NaN,36.0,177.0,150.0,185.0,NaN,2026-04-02 23:10:46.525035,Burari Crossing


In [18]:
print("\nNull counts:")
print(df.isnull().sum())

print("\nRows with PM2.5 null:")
print(df[df["PM2.5"].isnull()])


Null counts:
id               0
station          0
last_update      0
CO               0
NH3              7
NO2              0
OZONE            1
PM10             1
PM2.5            1
SO2              7
processed_at     0
station_clean    0
dtype: int64

Rows with PM2.5 null:
    id                      station          last_update    CO  NH3  NO2  \
27  28  New Moti Bagh, Delhi - MHUA  02-04-2026 16:00:00  46.0  NaN  9.0   

    OZONE  PM10  PM2.5  SO2               processed_at  station_clean  
27    NaN   NaN    NaN  NaN 2026-04-02 23:10:46.525035  New Moti Bagh  


In [19]:
# drop rows where target is null
df_clean = df.dropna(subset=["PM2.5"])
print("Before:", df.shape)
print("After drop:", df_clean.shape)

Before: (43, 12)
After drop: (42, 12)


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import train_test_split, cross_val_score
import numpy as np

In [ ]:
pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),   # 1. handle NaN
    ("scaler", StandardScaler()),                  # 2. scale features
    ("model", RidgeCV(alphas=[0.1, 1.0, 10.0, 100.0]))  # 3. Ridge with auto alpha selection
])

In [ ]:
features = ["CO", "NH3", "NO2", "OZONE", "PM10", "SO2"]
target = "PM2.5"

X = df_clean[features]
y = df_clean[target]

# 5-fold CV instead of single split (dataset is too small for hold-out)
cv_scores = cross_val_score(pipeline, X, y, cv=5, scoring="r2")

print("CV R² scores:", cv_scores.round(3))
print("Mean R²    :", np.mean(cv_scores).round(3))
print("Std R²     :", np.std(cv_scores).round(3))

In [ ]:
# Fit on full data to inspect best alpha chosen by RidgeCV
pipeline.fit(X, y)
print("Best alpha selected by RidgeCV:", pipeline.named_steps["model"].alpha_)

In [38]:
df.head()

,id,station,last_update,CO,NH3,NO2,OZONE,PM10,PM2.5,SO2,processed_at,station_clean
0,1,"Alipur, Delhi - DPCC",02-04-2026 16:00:00,28.0,14.0,27.0,47.0,135.0,131.0,11.0,2026-04-02 23:10:46.525035,Alipur
1,2,"Anand Vihar, Delhi - DPCC",02-04-2026 16:00:00,68.0,6.0,31.0,51.0,248.0,220.0,28.0,2026-04-02 23:10:46.525035,Anand Vihar
2,3,"Ashok Vihar, Delhi - DPCC",02-04-2026 16:00:00,38.0,7.0,51.0,151.0,167.0,173.0,22.0,2026-04-02 23:10:46.525035,Ashok Vihar
3,4,"Bawana, Delhi - DPCC",02-04-2026 16:00:00,65.0,7.0,27.0,202.0,171.0,173.0,19.0,2026-04-02 23:10:46.525035,Bawana
4,5,"Burari Crossing, Delhi - IMD",02-04-2026 16:00:00,29.0,NaN,36.0,177.0,150.0,185.0,NaN,2026-04-02 23:10:46.525035,Burari Crossing


## 3 Model Comparison — Linear Regression vs Random Forest vs XGBoost
### Data: aqi_processed_full (23,306 rows — historical + real merged)

In [ ]:
import pandas as pd
import numpy as np
import psycopg2
from dotenv import load_dotenv
import os

load_dotenv(dotenv_path="c:/applied-ml-apps/aqi-mlops/.env")
DATABASE_URL = os.getenv("DATABASE_URL")

if not DATABASE_URL:
    raise ValueError("DATABASE_URL not set in environment")

with psycopg2.connect(DATABASE_URL) as conn:
    df_full = pd.read_sql(
        "SELECT * FROM aqi_processed_full ORDER BY last_update ASC",
        conn
    )

print(f"Loaded: {df_full.shape[0]} rows x {df_full.shape[1]} columns")
print(f"Date range: {df_full['last_update'].min()} to {df_full['last_update'].max()}")
print(f"Sources: {df_full['source'].value_counts().to_dict()}")
df_full.head()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score, KFold
from xgboost import XGBRegressor
import numpy as np

features = ["CO", "NH3", "NO2", "OZONE", "PM10", "SO2"]
target   = "PM2.5"

# Drop rows where target is null
df_model = df_full.dropna(subset=[target]).reset_index(drop=True)

X = df_model[features]
y = df_model[target]

print(f"Training data: {X.shape[0]} rows x {X.shape[1]} features")
print(f"Target (PM2.5) — mean: {y.mean():.1f}, std: {y.std():.1f}\n")

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── Define 3 pipelines ────────────────────────────────────────────────────────

models = {
    "Linear Regression (Ridge)": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler",  StandardScaler()),
        ("model",   Ridge(alpha=10.0)),
    ]),
    "Random Forest": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler",  StandardScaler()),
        ("model",   RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)),
    ]),
    "XGBoost": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("scaler",  StandardScaler()),
        ("model",   XGBRegressor(n_estimators=100, learning_rate=0.1,
                                  random_state=42, verbosity=0, n_jobs=-1)),
    ]),
}

# ── Run 5-Fold CV for each model ──────────────────────────────────────────────

results = []
for name, pipeline in models.items():
    scores = cross_val_score(pipeline, X, y, cv=kf, scoring="r2")
    results.append({
        "Model":   name,
        "Fold 1":  round(scores[0], 4),
        "Fold 2":  round(scores[1], 4),
        "Fold 3":  round(scores[2], 4),
        "Fold 4":  round(scores[3], 4),
        "Fold 5":  round(scores[4], 4),
        "Mean R²": round(scores.mean(), 4),
        "Std R²":  round(scores.std(), 4),
    })
    print(f"{name:<30} Mean R²={scores.mean():.4f}  Std={scores.std():.4f}")

print("\nDone.")

In [ ]:
# ── Summary Table ─────────────────────────────────────────────────────────────
df_results = pd.DataFrame(results)
df_results = df_results.sort_values("Mean R²", ascending=False).reset_index(drop=True)

print("=" * 65)
print("MODEL COMPARISON — 5-Fold CV R² Scores")
print("=" * 65)
print(df_results.to_string(index=False))
print("=" * 65)
print(f"\nBest model: {df_results.iloc[0]['Model']}")
print(f"Best Mean R²: {df_results.iloc[0]['Mean R²']}")